In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math

Note: you may need to restart the kernel to use updated packages.


: 

In [ ]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.

# Helper functions using quantum randomness (no Python RNG).
def quantum_random_bits(n, simulator, max_qubits=24):
    if n <= 0:
        return []
    bits = []
    remaining = n
    while remaining > 0:
        chunk = min(max_qubits, remaining)
        qc = QuantumCircuit(chunk, chunk)
        qc.h(range(chunk))
        qc.measure(range(chunk), range(chunk))
        compiled = transpile(qc, simulator)
        result = simulator.run(compiled, shots=1).result()
        counts = result.get_counts()
        bitstring = next(iter(counts))
        bitstring = bitstring[::-1]  # Align bit 0 with qubit 0.
        bits.extend(int(b) for b in bitstring)
        remaining -= chunk
    return bits


def measure_bit(prep_bit, prep_basis, meas_basis, simulator):
    qc = QuantumCircuit(1, 1)
    if prep_bit == 1:
        qc.x(0)
    if prep_basis == 1:
        qc.h(0)
    if meas_basis == 1:
        qc.h(0)
    qc.measure(0, 0)
    compiled = transpile(qc, simulator)
    result = simulator.run(compiled, shots=1).result()
    counts = result.get_counts()
    measured = int(next(iter(counts)))
    return measured


def sift_key(alice_bits, alice_bases, bob_bits, bob_bases):
    indices = [i for i in range(len(alice_bits)) if alice_bases[i] == bob_bases[i]]
    alice_key = [alice_bits[i] for i in indices]
    bob_key = [bob_bits[i] for i in indices]
    return indices, alice_key, bob_key


def error_rate(alice_key, bob_key):
    if not alice_key:
        return 0.0
    mismatches = sum(1 for a, b in zip(alice_key, bob_key) if a != b)
    return mismatches / len(alice_key)


In [ ]:
simulator = BasicSimulator()
num_qubits = 64

# Alice: choose random bits and bases, then prepare qubits accordingly.
alice_bits = quantum_random_bits(num_qubits, simulator)
alice_bases = quantum_random_bits(num_qubits, simulator)

# Bob: choose random bases, then measure each incoming qubit.
bob_bases = quantum_random_bits(num_qubits, simulator)
bob_measurements = []
for i in range(num_qubits):
    bob_measurements.append(
        measure_bit(alice_bits[i], alice_bases[i], bob_bases[i], simulator)
    )

# Sifting: keep positions where Alice and Bob used the same basis.
sifted_indices, alice_key, bob_key = sift_key(
    alice_bits, alice_bases, bob_measurements, bob_bases
)
err = error_rate(alice_key, bob_key)
discard_rate = 1.0 - (len(alice_key) / num_qubits) if num_qubits > 0 else 0.0
    
print("BB84 (no attacker)")
print(f"Raw qubits: {num_qubits}")
print(f"Sifted key length: {len(alice_key)}")
print(f"Discard rate: {discard_rate:.2f}")
print(f"Error rate in sifted key: {err:.2f}")


BB84 (no attacker)
Raw qubits: 64
Sifted key length: 31
Discard rate: 0.52
Error rate in sifted key: 0.00


In [ ]:
print(f"First 16 sifted key bits (Alice): {alice_key[:16]}")
print(f"First 16 sifted key bits (Bob):   {bob_key[:16]}")


First 16 sifted key bits (Alice): [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1]
First 16 sifted key bits (Bob):   [0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 1]
